# GoPay Google Play Review - Sentiment Analysis

**Author:** Muhammad Razan Parisya Putra  
**Notebook:** `04 - Sentiment Analysis`

This notebook continues from the preprocessing covered in [3-Gopay-Review-Preprocessing.ipynb](https://colab.research.google.com/drive/1Iwg8LQ69nvhEtv6wE4Ou7uM2hySBrhKd?usp=sharing)

## Dataset Loading

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

df_gopay_clean = pd.read_csv('/content/drive/MyDrive/Tugas 1/Dataset/gopay_reviews_clean.csv')
df_gopay_clean.head()

In [ ]:
df_gopay_clean.info()

## TextBlob Installation & Sentiment Scoring

In [ ]:
!pip install textblob -q

In [ ]:
from textblob import TextBlob

# Hitung polarity dan subjectivity dari final_text (teks yang sudah di-preprocessing)
df_gopay_clean['sentiment_polarity'] = df_gopay_clean['final_text'].astype(str).apply(
    lambda x: TextBlob(x).polarity
)
df_gopay_clean['sentiment_subjectivity'] = df_gopay_clean['final_text'].astype(str).apply(
    lambda x: TextBlob(x).subjectivity
)

print('Sentiment polarity dan subjectivity berhasil dihitung.')
df_gopay_clean[['content', 'final_text', 'sentiment_polarity', 'sentiment_subjectivity']].head(5)

In [ ]:
# Sentiment label sudah ada dari preprocessing (berdasarkan score rating)
# Positive: score 4-5 | Neutral: score 3 | Negative: score 1-2
print('Distribusi sentiment label:')
print(df_gopay_clean['sentiment'].value_counts())
print()
print('Proporsi (%):')
print((df_gopay_clean['sentiment'].value_counts(normalize=True) * 100).round(2))

In [ ]:
df_gopay_clean[['final_text', 'score', 'sentiment_polarity', 'sentiment_subjectivity', 'sentiment']].head(10)

## EDA — Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import seaborn as sns
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected=True)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df_gopay_clean.sample(5)

In [ ]:
df_gopay_clean.info()

In [ ]:
df_gopay_clean.isnull().sum()

In [ ]:
import missingno as msno
msno.matrix(df_gopay_clean)
plt.show()

In [ ]:
df_gopay_clean['at'] = pd.to_datetime(df_gopay_clean['at'])
df_gopay_clean['year'] = df_gopay_clean['at'].dt.year

df_selected = df_gopay_clean[['content', 'final_text', 'score', 'sentiment', 'at', 'year']]
df_selected.head()

### Scatter Plot: Polarity vs Subjectivity

In [ ]:
plt.figure(figsize=(15, 10))
sns.scatterplot(
    x=df_gopay_clean['sentiment_polarity'],
    y=df_gopay_clean['sentiment_subjectivity'],
    hue=df_gopay_clean['sentiment'],
    edgecolor='white',
    palette='pastel'
)
plt.title('Google Play Store GoPay Reviews — Sentiment Analysis (Polarity vs Subjectivity)', fontsize=16)
plt.xlabel('Sentiment Polarity')
plt.ylabel('Sentiment Subjectivity')
plt.show()

### Fungsi: Most Frequent Words

In [ ]:
from nltk.probability import FreqDist
import nltk
nltk.download('punkt', quiet=True)

def freq_words(x, terms=30):
    all_words = ' '.join([str(text) for text in x if isinstance(text, str)])
    all_words = all_words.split()
    fdist = FreqDist(all_words)
    words_df = pd.DataFrame({'word': list(fdist.keys()), 'count': list(fdist.values())})
    d = words_df.nlargest(columns='count', n=terms)
    plt.figure(figsize=(20, 5))
    ax = sns.barplot(data=d, x='word', y='count', palette='rainbow')
    ax.set(ylabel='Count')
    plt.xticks(rotation=45, ha='right')
    plt.title(f'Top {terms} Most Frequent Words')
    plt.tight_layout()
    plt.show()

In [ ]:
freq_words(df_gopay_clean['final_text'])

### Word Cloud — All Reviews

In [ ]:
all_text = ' '.join(df_gopay_clean['final_text'].dropna().astype(str))

wordcloud = WordCloud(
    width=800, height=400,
    background_color='white',
    collocations=False
).generate(all_text)

plt.figure(figsize=(10, 5), facecolor=None)
plt.imshow(wordcloud)
plt.axis('off')
plt.tight_layout(pad=0)
plt.show()

### Word Cloud — Positive vs Negative

In [ ]:
positive_reviews = df_gopay_clean[df_gopay_clean['sentiment'] == 'positive']['final_text'].astype(str)
negative_reviews = df_gopay_clean[df_gopay_clean['sentiment'] == 'negative']['final_text'].astype(str)

positive_wc = WordCloud(width=800, height=400, background_color='white', colormap='Greens').generate(' '.join(positive_reviews))
negative_wc = WordCloud(width=800, height=400, background_color='white', colormap='Reds').generate(' '.join(negative_reviews))

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.imshow(positive_wc, interpolation='bilinear')
plt.title('Positive Sentiment Word Cloud', fontsize=13, fontweight='bold')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(negative_wc, interpolation='bilinear')
plt.title('Negative Sentiment Word Cloud', fontsize=13, fontweight='bold')
plt.axis('off')

plt.tight_layout()
plt.show()

### Review Count by Year

In [ ]:
reviews_by_year = df_gopay_clean.groupby('year')['content'].count()

plt.figure(figsize=(12, 6))
ax = sns.barplot(x=reviews_by_year.index, y=reviews_by_year.values, palette='viridis')
plt.xlabel('Year')
plt.ylabel('Number of Reviews')
plt.title('Number of Reviews by Year — GoPay')
plt.xticks(rotation=45)

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')
plt.tight_layout()
plt.show()

### Distribution of Ratings

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x='score', data=df_gopay_clean, palette='viridis')
plt.title('Distribution of Ratings — GoPay')
plt.xlabel('Rating (1-5)')
plt.ylabel('Number of Reviews')
plt.show()

### Sentiment Trend Over Time (by Year)

In [ ]:
sentiment_over_time = df_gopay_clean.groupby(['year', 'sentiment']).size().unstack(fill_value=0)

sentiment_columns_order = ['negative', 'neutral', 'positive']
for col in sentiment_columns_order:
    if col not in sentiment_over_time.columns:
        sentiment_over_time[col] = 0
sentiment_over_time = sentiment_over_time[sentiment_columns_order]

stacked_bar_colors = {'negative': '#f57c73', 'neutral': '#f6ac69', 'positive': '#a0ced9'}
plot_bar_colors = [stacked_bar_colors[col] for col in sentiment_over_time.columns]

fig, ax = plt.subplots(figsize=(14, 7))
sentiment_over_time.plot(kind='bar', stacked=True, ax=ax,
                          color=plot_bar_colors, edgecolor='white')

ax.set_xlabel('Year')
ax.set_ylabel('Number of Reviews')
ax.set_title('Sentiment Distribution Over Time — GoPay Reviews', fontsize=13, fontweight='bold')
ax.legend(title='Sentiment')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Save Dataset with Sentiment Scores

In [ ]:
output_path = '/content/drive/MyDrive/Tugas 1/Dataset/gopay_reviews_sentiment.csv'
df_gopay_clean.to_csv(output_path, index=False)

print(f'Dataset saved: {output_path}')
print(f'Shape: {df_gopay_clean.shape}')
df_gopay_clean.head()